# 12 - Board Pack Reviewer Agent
## Executive Critique + Structured Risk Prioritization
20 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/12-board-pack-reviewer/board_pack_workbook.ipynb)

Board packs arrive full of management language: "challenging conditions", "positive
momentum", "under review". A non-executive director's job is to see through that
framing and identify what actually matters for governance.

This example builds an agent that reads a board pack and outputs a structured
`DirectorBriefing` -- not a summary, but an independent critique with risks ranked,
information gaps named, and probing questions for management.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The NED framing problem |
| 2 | Schema: TopRisk, InformationGap, DecisionRequired, DirectorBriefing |
| 3 | System prompt design for executive critique |
| 4 | Run on a realistic board pack |
| 5 | Interpreting the output |
| * | Exercises + Answer Key |

### Prerequisites
- Python 3.10+, or Google Colab
- OPENAI_API_KEY in .env or Colab Secrets

In [ ]:
try:
    from google.colab import userdata  # noqa: F401
    import subprocess
    subprocess.run(["pip", "install", "-q", "langchain-openai", "pydantic", "python-dotenv"], check=True)
    print("Packages installed.")
except ImportError:
    print("Local environment.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if not os.getenv("OPENAI_API_KEY"):
        raise EnvironmentError("Set OPENAI_API_KEY in .env or Colab Secrets.")
print("API key ready.")

---

## Part 1 - The NED Framing Problem

Management writes board packs. NEDs read them. That asymmetry creates risk:

- **Omission**: Issues exist but are not disclosed
- **Framing**: Risks described as "being managed" without specifics
- **Timing**: Decisions presented for approval without adequate preparation time
- **Complexity**: Information buried or formatted to obscure rather than inform

A good NED briefing reframes the pack content from the *board's* perspective:
- What are the top risks *as governance concerns*?
- What material information is *missing* from this pack?
- What is the board actually being *asked to decide*?
- What questions should management *not be allowed to duck*?

That is what `DirectorBriefing` encodes.

---

## Part 2 - Schema

In [ ]:
from typing import List, Literal, Optional
from pydantic import BaseModel, Field


class TopRisk(BaseModel):
    rank: int = Field(description="Priority rank: 1 is the highest concern")
    area: Literal[
        "financial", "operational", "strategic", "regulatory", "governance", "reputational"
    ]
    severity: Literal["critical", "high", "medium"]
    title: str
    detail: str = Field(
        description="What specifically is the risk and why it matters to the board"
    )
    source_section: str = Field(
        description="Which section of the board pack this comes from"
    )
    suggested_question: str = Field(
        description="Question a non-executive director should ask management on this risk"
    )


class InformationGap(BaseModel):
    section: str = Field(description="Which section has the gap")
    missing: str = Field(description="What information is absent or unclear")
    why_it_matters: str = Field(description="Why the board needs this before deciding")


class DecisionRequired(BaseModel):
    item: str = Field(
        description="The specific decision or approval the board is being asked to make"
    )
    context: str = Field(description="Background a director needs to vote on this item")
    recommendation: Optional[str] = Field(
        default=None,
        description="Management's stated recommendation, if provided"
    )
    key_consideration: str = Field(
        description="The one thing a director must weigh before approving"
    )


class DirectorBriefing(BaseModel):
    """Structured board pack review framed for a non-executive director."""

    company: Optional[str] = None
    meeting_date: Optional[str] = None
    overall_pack_quality: Literal["strong", "adequate", "weak"] = Field(
        description="Overall quality of the board pack as a governance document"
    )
    executive_assessment: str = Field(
        description="3-4 sentence summary for a NED arriving five minutes before the meeting"
    )
    top_risks: List[TopRisk] = Field(
        description="Up to five risks ranked by severity"
    )
    information_gaps: List[InformationGap] = Field(
        description="Material information absent from the pack"
    )
    decisions_required: List[DecisionRequired] = Field(
        description="Items requiring board approval or formal decision"
    )
    questions_for_management: List[str] = Field(
        description="Suggested questions -- probing, not procedural"
    )

print("Schema defined.")
print("Output: DirectorBriefing (risks + gaps + decisions + questions)")

---

## Part 3 - System Prompt Design

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

REVIEWER_SYSTEM = SystemMessage(
    """You are an experienced non-executive director reviewing a board pack before a meeting.

Your job is to produce a structured briefing that helps fellow directors cut through
management language and focus on what actually matters.

Rules:
- Frame every risk as a board concern, not a management update
- Name information gaps explicitly -- "the pack does not disclose X" is more
  useful than silence
- Questions for management must be probing: challenge assumptions, not process
- overall_pack_quality reflects governance fitness, not length or formatting
- If something looks like it has been sanitised or is missing context, say so

You serve shareholders and stakeholders, not management."""
)

print("Key prompt design choices:")
print("  1. Role = NED, not analyst -- changes the lens entirely")
print("  2. Explicit instruction to name gaps -- silence is the default")
print("  3. Questions must be probing, not procedural")
print("  4. Pack quality is governance fitness, not production quality")
print("  5. Loyalty to shareholders, not management -- breaks sycophancy")

---

## Part 4 - Run on a Realistic Board Pack

In [ ]:
BOARD_PACK = """NEXUS RETAIL GROUP PLC\nBOARD OF DIRECTORS MEETING\n12 March 2025\n\nITEM 2: CEO UPDATE\nTrading conditions in Q4 2024 were challenging. Digital channel grew 22% YoY,\nnow 31% of total revenue. CFO Sarah Chen resigned January 2025. Interim CFO\nMark Patel appointed 3 February 2025. Permanent search ongoing.\n\nITEM 3: FINANCIAL PERFORMANCE\nRevenue: GBP 2.41bn (-3.2% vs FY2023) | Gross margin: 28.1% (-1.4pp)\nEBITDA: GBP 89m (-18%) | Net debt: GBP 312m | Leverage: 3.5x EBITDA\nInterest cover: 2.1x (covenant minimum: 2.0x)\nFY2024 audit not yet complete. Numbers subject to change.\nQ1 2025: LFL sales -1.8% | Digital +19% | Q1 gross margin not provided.\nBoard asked to approve dividend of 12p per share.\n\nITEM 4: PROPOSED ACQUISITION -- FRESHLOCAL LTD\nConsideration: GBP 95m (GBP 85m cash, GBP 10m deferred).\nFreshLocal FY2024: Revenue GBP 38m, EBITDA GBP 2.1m (5.5% margin).\nMultiple: 45x EV/EBITDA. No integration plan. No synergy case. No funding plan.\nBoard asked to approve acquisition in principle.\n\nITEM 5: AUDIT & RISK COMMITTEE REPORT\nKPMG going concern emphasis of matter paragraph possible. No further detail.\nData breach Nov 2024: 180,000 customer payment records exposed. ICO notified.\nGBP 2.1m provision raised. Regulatory outcome pending.\nWhistleblower report Q4 2024 re procurement practices. Under investigation."""

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
reviewer = REVIEWER_SYSTEM | llm.with_structured_output(DirectorBriefing)

briefing = reviewer.invoke(
    HumanMessage(content="Board pack to review:\n\n" + BOARD_PACK)
)

print(f"Company: {briefing.company}")
print(f"Meeting: {briefing.meeting_date}")
print(f"Pack quality: {briefing.overall_pack_quality.upper()}")

---

## Part 5 - Interpreting the Output

In [ ]:
SEV = {"critical": "CRIT", "high": "HIGH", "medium": "MED "}

print("=" * 65)
print("EXECUTIVE ASSESSMENT")
print("=" * 65)
print(briefing.executive_assessment)

print(f"\nTOP RISKS ({len(briefing.top_risks)}):")
for r in briefing.top_risks:
    print(f"\n  [{r.rank}] [{SEV[r.severity]}] [{r.area.upper()}] {r.title}")
    print(f"  Source: {r.source_section}")
    print(f"  {r.detail}")
    print(f"  Q: {r.suggested_question}")

In [ ]:
if briefing.information_gaps:
    print(f"INFORMATION GAPS ({len(briefing.information_gaps)}):")
    for g in briefing.information_gaps:
        print(f"\n  [{g.section}] {g.missing}")
        print(f"  Why it matters: {g.why_it_matters}")

if briefing.decisions_required:
    print(f"\nDECISIONS REQUIRED ({len(briefing.decisions_required)}):")
    for d in briefing.decisions_required:
        print(f"\n  DECISION: {d.item}")
        print(f"  Context: {d.context}")
        if d.recommendation:
            print(f"  Mgmt recommends: {d.recommendation}")
        print(f"  Key consideration: {d.key_consideration}")

if briefing.questions_for_management:
    print("\nSUGGESTED QUESTIONS FOR MANAGEMENT:")
    for i, q in enumerate(briefing.questions_for_management, 1):
        print(f"  {i}. {q}")

---

## Exercise 1 - Add Pack Quality Scoring Per Section

The `overall_pack_quality` field gives a single rating. Add per-section quality scoring
so the board knows *which* sections are weak, not just that the pack overall is.

```python
class SectionQuality(BaseModel):
    section: str
    quality: Literal["strong", "adequate", "weak", "missing"]
    comment: str
```

Add `section_quality: List[SectionQuality]` to `DirectorBriefing` and instruct
the model to rate each section separately.

In [ ]:
# Exercise 1 Starter
class SectionQuality(BaseModel):
    section: str
    quality: Literal["strong", "adequate", "weak", "missing"]
    comment: str

# TODO: Create DetailedBriefing that extends DirectorBriefing with
#       section_quality: List[SectionQuality]
# TODO: Update REVIEWER_SYSTEM to instruct per-section quality assessment
# TODO: Run and print section quality table

In [ ]:
# Exercise 1 Answer Key
class DetailedBriefing(BaseModel):
    company: Optional[str] = None
    meeting_date: Optional[str] = None
    overall_pack_quality: Literal["strong", "adequate", "weak"]
    executive_assessment: str
    top_risks: List[TopRisk]
    information_gaps: List[InformationGap]
    decisions_required: List[DecisionRequired]
    questions_for_management: List[str]
    section_quality: List[SectionQuality]


DETAILED_SYSTEM = SystemMessage(
    """You are an experienced non-executive director reviewing a board pack.

Produce a DirectorBriefing as instructed. Additionally, rate each identifiable
section of the board pack on quality (strong/adequate/weak/missing) and explain
the rating in one sentence.

You serve shareholders and stakeholders, not management."""
)

detailed_reviewer = DETAILED_SYSTEM | llm.with_structured_output(DetailedBriefing)
detailed_briefing = detailed_reviewer.invoke(
    HumanMessage(content="Board pack to review:\n\n" + BOARD_PACK)
)

QUAL = {"strong": "STRONG", "adequate": "OK    ", "weak": "WEAK  ", "missing": "ABSENT"}
print("SECTION QUALITY:")
for s in detailed_briefing.section_quality:
    print(f"  [{QUAL[s.quality]}] {s.section}")
    print(f"           {s.comment}")

---

## What You Built

A board pack reviewer demonstrating executive critique + structured risk prioritization:

1. **NED framing** in the system prompt forces the model to challenge management
   language rather than summarise it
2. **Explicit information gap field** -- the model is instructed to name what's
   absent, not just what's present
3. **Decision identification** separates what the board is being asked to *approve*
   from what it is being asked to *note*
4. **Probing questions per risk** give the NED a tool for the meeting, not just a
   reading list

The pattern scales: feed multiple board pack sections as separate chunks, run
per-section extraction (like example 10's document pipeline), then synthesise
into one `DirectorBriefing`.

---
*Example 12 of 23 - agent-use-cases*